# Hydranomic

Execute as células na ordem.

In [6]:
from pathlib import Path
import json

def localizar_projeto():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / 'hydranomic' / 'workflows').is_dir():
            return candidato
    raise FileNotFoundError('Pasta hydranomic/workflows não encontrada.')

PROJETO = localizar_projeto()
PASTA_WORKFLOWS = PROJETO / 'hydranomic' / 'workflows'
ARQUIVO_CONTAS = PASTA_WORKFLOWS / 'conta_povoada.json'
ARQUIVO_CONFIG = PASTA_WORKFLOWS / 'config.json'
ARQUIVO_GRAFICO = PASTA_WORKFLOWS / 'grafico_anual.json'
with ARQUIVO_CONTAS.open(encoding='utf-8') as arquivo:
    dados = json.load(arquivo)
print(f'Projeto: {PROJETO}')
print(f'Registros carregados: {len(dados)}')

Projeto: c:\Users\Aluno\Repositorio\Hydranomic
Registros carregados: 36


In [ ]:
# data_chart.py
from datetime import datetime

def gerar_dados_grafico(data, ano_referencia=None):
    if ano_referencia is None:
        ano_referencia = datetime.now().year
    consumo_mensal = {}
    mes_atual = None
    for registro in data:
        partes = str(registro.get('id', '')).split('-')
        if len(partes) != 2:
            continue
        try:
            ano_registro, mes_registro = map(int, partes)
        except ValueError:
            continue
        if ano_registro == ano_referencia and 1 <= mes_registro <= 12:
            consumo_mensal[mes_registro] = registro.get('consumo_m3')
            mes_atual = mes_registro
    meses_tags = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
    return {
        'ano_referencia': ano_referencia,
        'mes_atual': mes_atual,
        'unidade_medida': 'm³',
        'meses': [{'mes': mes, 'rotulo': meses_tags[mes - 1], 'consumo_m3': consumo_mensal.get(mes)} for mes in range(1, 13)],
    }

In [4]:
# machinelearning.py
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

TARIFA_ATUAL = 59.6

def preparar_modelo(data):
    dataframe = pd.DataFrame(data).sort_values(['unidade_consumidora', 'id']).reset_index(drop=True)
    partes_id = dataframe['id'].astype(str).str.split('-', expand=True)
    dataframe['ano'] = partes_id[0].astype(int)
    dataframe['mes'] = partes_id[1].astype(int)
    ano_inicial = dataframe['ano'].min()
    mes_inicial = dataframe.loc[dataframe['ano'] == ano_inicial, 'mes'].min()
    dataframe['tempo_iniciofinal'] = (dataframe['ano'] - ano_inicial) * 12 + dataframe['mes'] - mes_inicial
    dataframe['memoria_consumo'] = dataframe.groupby('unidade_consumidora')['consumo_m3'].shift(1)
    treino = dataframe.dropna(subset=['memoria_consumo'])
    entradas = ['mes', 'tempo_iniciofinal', 'memoria_consumo']
    modelo = RandomForestRegressor(n_estimators=250, random_state=45)
    modelo.fit(treino[entradas], treino['consumo_m3'])
    return dataframe, modelo, entradas

def iniciar_ml(data, tarifa=TARIFA_ATUAL):
    dataframe, modelo, entradas = preparar_modelo(data)
    ultima_linha = dataframe.iloc[-1]
    ultimo_mes, ultimo_ano = int(ultima_linha['mes']), int(ultima_linha['ano'])
    proximo_mes, proximo_ano = (1, ultimo_ano + 1) if ultimo_mes == 12 else (ultimo_mes + 1, ultimo_ano)
    dados_previsao = pd.DataFrame([[proximo_mes, ultima_linha['tempo_iniciofinal'] + 1, ultima_linha['consumo_m3']], columns=entradas)
    previsao_m3 = float(modelo.predict(dados_previsao)[0])
    return {'ano': proximo_ano, 'mes': proximo_mes, 'unidade_consumidora': ultima_linha['unidade_consumidora'], 'previsao_m3': previsao_m3, 'tarifa': tarifa, 'valor_conta': previsao_m3 * tarifa}

SyntaxError: closing parenthesis ')' does not match opening parenthesis '[' (4128170042.py, line 27)

In [ ]:
# metas.py
def carregar_config(caminho_config=ARQUIVO_CONFIG):
    caminho_config = Path(caminho_config)
    if not caminho_config.exists() or caminho_config.stat().st_size == 0:
        return {'meta_ativa': None, 'solicitacao_exclusao': None}
    with caminho_config.open(encoding='utf-8') as arquivo:
        config = json.load(arquivo)
    config.setdefault('meta_ativa', None)
    config.setdefault('solicitacao_exclusao', None)
    return config

def salvar_config(config, caminho_config=ARQUIVO_CONFIG):
    caminho_config = Path(caminho_config)
    caminho_config.parent.mkdir(parents=True, exist_ok=True)
    with caminho_config.open('w', encoding='utf-8') as arquivo:
        json.dump(config, arquivo, ensure_ascii=False, indent=2)

def criar_meta(tipo, nome, valor, periodo, descricao='', caminho_config=ARQUIVO_CONFIG):
    config = carregar_config(caminho_config)
    if config['meta_ativa'] is not None:
        raise ValueError('Já existe uma meta ativa ou aguardando exclusão.')
    try:
        valor = float(valor)
    except (TypeError, ValueError) as erro:
        raise ValueError('O valor da meta deve ser numérico.') from erro
    if not nome or valor <= 0 or not periodo:
        raise ValueError('Nome, valor positivo e período são obrigatórios.')
    config['meta_ativa'] = {'tipo': tipo, 'nome': nome, 'valor': valor, 'periodo': periodo, 'descricao': descricao, 'status': 'ativa'}
    config['solicitacao_exclusao'] = None
    salvar_config(config, caminho_config)
    return config['meta_ativa']

def solicitar_exclusao_meta(caminho_config=ARQUIVO_CONFIG):
    config = carregar_config(caminho_config)
    if config['meta_ativa'] is None:
        raise ValueError('Não existe uma meta para excluir.')
    config['solicitacao_exclusao'] = {'status': 'pendente', 'meta': config['meta_ativa']}
    config['meta_ativa']['status'] = 'exclusao_solicitada'
    salvar_config(config, caminho_config)
    return config['solicitacao_exclusao']

def confirmar_exclusao_meta(caminho_config=ARQUIVO_CONFIG):
    config = carregar_config(caminho_config)
    if config['solicitacao_exclusao'] is None:
        raise ValueError('Não existe uma solicitação de exclusão pendente.')
    config['meta_ativa'] = None
    config['solicitacao_exclusao'] = None
    salvar_config(config, caminho_config)
    return config

In [1]:
# app.py - integração
previsao = iniciar_ml(dados)
grafico = gerar_dados_grafico(dados, ano_referencia=previsao['ano'])
configuracao = carregar_config()
with ARQUIVO_GRAFICO.open('w', encoding='utf-8') as arquivo:
    json.dump(grafico, arquivo, ensure_ascii=False, indent=2)
resultado = {'previsao': previsao, 'grafico': grafico, 'meta_ativa': configuracao['meta_ativa']}
resultado

NameError: name 'iniciar_ml' is not defined